# Make Links — the link graphs and Link_Check

Reads only the base graphs (`base/<source>/<source>.ttl`; the SDTMIG and SDTM Model graphs are built locally by their readers). Nothing is read from a source file.

1. the link specs of `schema/links.yaml`, checked (`cfa.link.load_links`); the link shapes in `schema/shacl/links.shacl.ttl` (written by `checks/00_Schema.ipynb`) must be current;
2. the base graphs, as one graph;
3. per link, its triples: a row whose key value(s) equal the target columns of exactly one target row gets one triple, row → target row. Published-key links go to the named graph `links`, links by rule to `by-rule`;
4. the link shapes (SHACL-SPARQL, sh:Info; focus nodes by a SHACL-AF SPARQL target, so pySHACL runs with advanced=True) over base + links: every row with a value in a link column and no link (for a column of several keys: fewer links than keys) is a result, and so is a linked row whose target carries a marker stated in links.yaml (`flag`). The report is Link_Check;
5. writes the links by a rule that matches a general name (the links with a rule function) as SSSOM mapping sets, `by-rule.sssom.tsv` and `by-rule-signin.sssom.tsv`: one row per link triple, the predicate by the part of the rule that matched (skos:exactMatch where the names are the same, skos:broadMatch where a general name matched), mapping_justification semapv:LexicalMatching, and that part of the rule in the comment. Read back, each file gives the triples of its links in the link graph;
6. writes the link graphs (longturtle) and their hashes, `Link_Check.xlsx` rendered from the report, `Provenance.json`.

Links that start from or end at a row of a sign-in source (SDTMIG, SDTM Model) are written to separate `-signin` files (graph and SSSOM), not published (`.gitignore`), until decided otherwise.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import pyshacl
from rdflib import Graph

from cfa import SCHEMA_DIR
from cfa.graph import graph_sha256, write_longturtle
from cfa.link import link_check_frame, link_graph_iri, link_shapes, links_sheet, load_links, make_links
from cfa.sssom import SEMAPV, SKOS, expand, mapping_rows, read_sssom, write_sssom
from cfa.pins import file_sha256
from cfa.provenance import TOOLS, flat
from cfa.read import read_xlsx_table
from cfa.view import TEXT_HASH_METHOD, text_sha256, write_view
from importlib.metadata import version

LINKS = SCHEMA_DIR / 'links.yaml'
SHAPES = SCHEMA_DIR / 'shacl' / 'links.shacl.ttl'
OUT = REPO_ROOT / 'linked'
SOURCES = ['nsv-registry', 'cosmos', 'sdtmig', 'sdtm-model', 'sdtm-ct', 'ncit-flat', 'nci-code-cui-map']
SIGNIN = {'sdtmig', 'sdtm-model'}

specs = load_links(LINKS)
shapes = link_shapes(specs)
assert graph_sha256(Graph().parse(SHAPES)) == graph_sha256(shapes), 'link shapes not current: run checks/00_Schema'
print(len(specs), 'links; shapes current')

# Prefixes for the written files: one per link namespace, named {source}_{table}, and ncit for NCI's IRIs.
# Binding them all makes the text deterministic (rdflib would otherwise invent ns1, ns2, ... in no fixed order).
PREFIXES = {'ncit': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#'}
for spec in specs:
    if spec['kind'] == 'published key':
        PREFIXES[f"{spec['domain_source']}_{spec['domain'].replace(' ', '_')}"] = spec['iri'].split('#')[0] + '#'
    else:
        # a by-rule IRI ends in /{target source}/{target table}: one prefix per link, up to the last "/"
        column = ''.join(ch if ch.isalnum() else '_' for ch in spec['columns'][-1])
        PREFIXES[f"rule_{spec['domain_source']}_{spec['domain'].replace(' ', '_')}_{column}_{spec['range_source']}"] = spec['iri'].rsplit('/', 1)[0] + '/'

## 2. The base graphs

In [ ]:
base = Graph()
base_files = {}
for src in SOURCES:
    path = REPO_ROOT / 'base' / src / f'{src}.ttl'
    base_files[path.relative_to(REPO_ROOT).as_posix()] = file_sha256(path)
    before = len(base)
    base.parse(path)
    print(f'{src}: {len(base) - before} triples')

## 3. The links

One file per named graph; a link that touches a sign-in source goes to the `-signin` file of its graph. The links are made in the order of links.yaml, so that a link with `unless` sees the earlier links.

In [ ]:
parts = {'links': Graph(), 'by-rule': Graph(), 'links-signin': Graph(), 'by-rule-signin': Graph()}
made = Graph()
matched = {'by-rule': [], 'by-rule-signin': []}  # per SSSOM file: (spec, the matched target key per triple)
for spec in specs:
    part = 'links' if spec['kind'] == 'published key' else 'by-rule'
    if {spec['domain_source'], spec['range_source']} & SIGNIN:
        part += '-signin'
    m = {} if spec['rule_function'] else None
    triples = make_links(base, spec, made, m)
    if m is not None:
        matched[part].append((spec, m))
    parts[part] += triples
    made += triples
    print(f"{spec['name']}: {len(triples)} -> {part}")

## 4. Link shapes over base + links: the report

In [ ]:
data = Graph()
data += base
for g in parts.values():
    data += g
conforms, report, _ = pyshacl.validate(data, shacl_graph=shapes, advanced=True, inplace=True)
check = link_check_frame(report, data, specs)
print('results:', len(check))
print(check.groupby('Link').size().to_string())

## 5. The links by a general-name rule as SSSOM

One mapping set per file, with the CURIE prefixes of the row tables it uses. The rule of each link is stated in `schema/links.yaml`; the comment of a row names the link and the part of its rule that matched.

In [ ]:
from collections import Counter

ID = 'https://kerfors.github.io/cdisc-for-all/id/'
ID_PREFIXES = {'cosmos_dss': ID + 'cosmos/cdisc_sdtm_dataset_specializations_latest/',
               'sdtmig': ID + 'sdtmig/SDTMIG/',
               'sdtm_model': ID + 'sdtm-model/SDTM/',
               'nsv_registry': ID + 'nsv-registry/Approved%20NSVs-alpha/'}
sssom_written = {}
for part, specs_matched in matched.items():
    if not specs_matched:
        continue
    rows = mapping_rows(specs_matched, ID_PREFIXES)
    used = sorted({r[c].split(':', 1)[0] for r in rows for c in ('subject_id', 'object_id')})
    curie_map = {'skos': SKOS, 'semapv': SEMAPV} | {p: ID_PREFIXES[p] for p in used}
    path = OUT / f'{part}.sssom.tsv'
    meta = {
        'curie_map': curie_map,
        'mapping_set_id': f'https://kerfors.github.io/cdisc-for-all/linked/{path.name}',
        'mapping_set_description': 'The cdisc-for-all links by a rule that matches a general name, one row per link triple: '
                                   + '; '.join(f"{s['name']} ({s['iri']})" for s, _ in specs_matched)
                                   + '. The rules are stated in schema/links.yaml.',
        'license': 'https://opensource.org/licenses/MIT',
        'mapping_tool': 'cdisc-for-all linked/Make_Links.ipynb (cfa.link, cfa.sssom)',
    }
    write_sssom(path, rows, meta)
    # read back: the rows, with the link predicate in place of the SSSOM predicate, are the triples of these links
    back_meta, back = read_sssom(path)
    assert back_meta == meta and back == rows
    iri = {s['name']: s['iri'] for s, _ in specs_matched}
    from_sssom = {(expand(r['subject_id'], curie_map), iri[r['comment'].split(':', 1)[0]], expand(r['object_id'], curie_map)) for r in back}
    in_graph = {(str(s), str(p), str(o)) for s, p, o in parts[part] if str(p) in iri.values()}
    assert from_sssom == in_graph and len(back) == len(in_graph)
    sssom_written[part] = {'file': path.name, 'rows': len(rows), 'sha256': file_sha256(path)}
    print(path.name, len(rows), 'rows; read back equal to the link triples;',
          dict(sorted(Counter(r['predicate_id'] for r in rows).items())))

## 6. Write the link graphs, Link_Check, Provenance

In [ ]:
OUT.mkdir(exist_ok=True)
written = {}
for part, g in parts.items():
    if len(g) == 0:
        continue
    path = OUT / f'{part}.ttl'
    write_longturtle(g, path, PREFIXES)
    back = Graph().parse(path)
    assert graph_sha256(back) == graph_sha256(g)
    written[part] = {'file': path.name, 'graph': link_graph_iri('published key' if part.startswith('links') else 'by rule'),
                     'triples': len(g), 'graph_sha256': graph_sha256(g)}
    print(path.name, len(g), written[part]['graph_sha256'])

KEYS = ['Link', 'Row', 'Value']
CHECK_HASH = text_sha256(check, KEYS)
prov = {
    'layer': 'linked',
    'inputs': base_files,
    'schema': {p.relative_to(REPO_ROOT).as_posix(): file_sha256(p) for p in (LINKS, SHAPES)},
    'link_graphs': written,
    'sssom': sssom_written,
    'graph_sha256_method': 'SHA-256 of the canonical N-Triples (RDFC-1.0; no blank nodes)',
    'link_check_text_sha256': CHECK_HASH,
    'text_sha256_method': TEXT_HASH_METHOD,
    'tools': {t: version(t) for t in TOOLS},
}
(OUT / 'Provenance.json').write_text(json.dumps(prov, indent=1, ensure_ascii=False) + '\n', encoding='utf-8')

readme = [
    'Link_Check: every row of a base graph that has a value in a link column and no link (or, for a column of several keys, fewer links than keys); and every linked row whose target carries a marker stated in links.yaml (column Flag of sheet Schema; the link stays).',
    'Rendered from the SHACL validation report of the link shapes (schema/shacl/links.shacl.ttl, generated from schema/links.yaml) over the base graphs and the link graphs.',
    'Broken links stay broken: they are reported here, not repaired. Nothing in this sheet is a value of ours: Value is the cell as published; Not resolved is that cell split at the separator, minus the keys that have a link.',
    'Sheet Schema: the links as stated in schema/links.yaml. Provenance: the base graphs read and the hashes.',
    'cdisc-for-all: https://github.com/kerfors/cdisc-for-all',
]
xlsx = OUT / 'Link_Check.xlsx'
write_view(xlsx, readme, {'Link_Check': check}, links_sheet(specs), flat(prov))
assert text_sha256(read_xlsx_table(xlsx, 'Link_Check'), KEYS) == CHECK_HASH
print(xlsx.name, 'read back: text hash equal', CHECK_HASH)